# Notebook 07b — Supervised 2D Encoder: Temperature Sweep
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

## Why this notebook exists

Notebook 07 (τ=0.07) **collapsed during training**: all 6,579 embeddings clustered in a thin arc spanning only ~38° of the unit circle. Probe accuracy 32.8% (vs 64D's 67.7%); val loss flat at log(64). This is an **optimization failure**, not evidence that BSISO is high-dimensional.

**Likely cause:** τ=0.07 is well-tuned for high-dim (S⁶³ in R⁶⁴), where there's lots of angular room. On S¹ in R² (after L2 normalization), every two random points are already within π — a sharp τ produces near-degenerate gradients and easy collapse.

**This notebook sweeps τ ∈ {0.2, 0.5, 1.0}** to test whether a softer temperature can train the same 2D encoder without collapse. Per-τ pipeline is identical to notebook 07; only τ changes.

## What gets produced

For each τ ∈ {0.2, 0.5, 1.0}:
- `checkpoints/encoder_2d_lee_tau{N}_final.pth`
- `checkpoints/training_history_2d_lee_tau{N}.json`
- `results/lee_2d_tau{N}/embeddings.npy`
- `results/lee_2d_tau{N}/training_curves.png`
- `results/lee_2d_tau{N}/embedding_2d_overview.png`
- `results/lee_2d_tau{N}/linear_probe_results.json`
- `results/lee_2d_tau{N}/enso_displacement.png`

Where `{N}` = 020, 050, 100 (i.e. `int(τ × 100)`).

Aggregate comparison (4-way, includes τ=0.07 from existing `results/lee_2d/`):
- `results/lee_2d_tau_sweep/comparison_table.csv`
- `results/lee_2d_tau_sweep/scatter_4panel.png`
- `results/lee_2d_tau_sweep/temperature_sweep_summary.md` (with auto-decision)

## Decision rule

For each τ we compute: BSISO phase val acc, ENSO displacement z-score, **angular spread** (range of embedding angles in radians, max 2π).

- **If any τ achieves val ≥ 62% AND z ≥ 3.0**: pick the best τ → greenlight Phase 2 with that τ.
- **If all τ collapse (angular spread < 1.0 rad)**: the issue isn't temperature — go to Phase 4 dim sweep or drop L2 norm (Option B from the diagnostic).
- **If some τ partially works** (spread > π but probe < 62%): document and decide based on closest-to-baseline.

## Runtime

3 × ~30 min on T4 ≈ 90 min. Fits in one Colab session.

---
## Before running
1. Notebook 07 should have run first (we load its τ=0.07 result for the comparison table).
2. Set runtime to T4 GPU.

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import matplotlib.pyplot as plt

# ── Sweep config ─────────────────────────────────────────────────────────────
EMBEDDING_DIM = 2
APPROACH      = 'lee'
TAU_VALUES    = [0.2, 0.5, 1.0]      # ← the sweep
EPOCHS        = 50
BATCH_SIZE    = 64
LR            = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
SWEEP_DIR      = f'{PROJECT_DIR}/results/lee_2d_tau_sweep'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SWEEP_DIR, exist_ok=True)

X_FILE      = 'X_MJJAS_lee.npy'
LABELS_FILE = 'labels_aligned_mjjas_lee.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:        {device}')
print(f'Embedding dim: {EMBEDDING_DIM}')
print(f'Sweep τ:       {TAU_VALUES}')
print(f'Epochs:        {EPOCHS}')
print(f'Sweep dir:     results/lee_2d_tau_sweep/')

def tau_tag(tau):
    """Filename-safe tau identifier: 0.2 → 'tau020', 0.5 → 'tau050', 1.0 → 'tau100'."""
    return f'tau{int(round(tau * 100)):03d}'

for t in TAU_VALUES:
    print(f'  τ={t:.2f}  →  {tau_tag(t)}')

## Cell 2 — Load Data + Year-Based Split + Phase-ENSO Index

Identical to notebook 07. Reused for all τ — data and split don't change.

In [ ]:
X      = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])

print(f'X shape:  {X.shape}  (N, channels, lat, lon)')
print(f'Labels:   {len(labels)} rows')

# Year-based split (deterministic — every 5th year held out)
all_years = sorted(labels['date'].dt.year.unique())
val_years = all_years[::5]
train_years = [y for y in all_years if y not in val_years]
year_col  = labels['date'].dt.year
train_idx = labels.index[year_col.isin(train_years)].values
val_idx   = labels.index[year_col.isin(val_years)].values

print(f'\nVal years ({len(val_years)}): {val_years}')
print(f'Train: {len(train_idx)} samples ({100*len(train_idx)/len(labels):.1f}%)')
print(f'Val:   {len(val_idx)} samples ({100*len(val_idx)/len(labels):.1f}%)')

# Phase-ENSO index for pair sampling
phase_enso_index = defaultdict(list)
for idx in train_idx:
    row = labels.loc[idx]
    if row['bsiso_amplitude'] > 1.0:
        key = (row['bsiso_phase'], row['enso_category'])
        phase_enso_index[key].append(idx)

active_total = sum(len(v) for v in phase_enso_index.values())
print(f'\nActive BSISO days in TRAIN (amplitude > 1.0): {active_total} / {len(train_idx)}')

## Cell 3 — PairSampler + Dataset + DataLoaders

Identical to notebook 07. Built once and reused across all τ runs.

In [ ]:
class PairSampler:
    def __init__(self, labels_df, phase_enso_index):
        self.labels = labels_df
        self.index  = phase_enso_index
        self.enso_categories = labels_df['enso_category'].unique().tolist()

    def sample_positive_pair(self):
        key = self._random_category()
        indices = self.index[key]
        if len(indices) < 2:
            return self.sample_easy_negative_pair()
        idx_A, idx_B = np.random.choice(indices, size=2, replace=False)
        year_A = self.labels.loc[idx_A, 'date'].year
        other = [i for i in indices if self.labels.loc[i, 'date'].year != year_A]
        if other:
            idx_B = np.random.choice(other)
        return idx_A, idx_B, 'positive'

    def sample_hard_negative_pair(self):
        if len(self.enso_categories) < 2:
            return self.sample_easy_negative_pair()
        phase = np.random.choice(range(1, 9))
        enso_A, enso_B = np.random.choice(self.enso_categories, size=2, replace=False)
        key_A, key_B = (phase, enso_A), (phase, enso_B)
        if not self.index[key_A] or not self.index[key_B]:
            return self.sample_positive_pair()
        idx_A = np.random.choice(self.index[key_A])
        idx_B = np.random.choice(self.index[key_B])
        return idx_A, idx_B, 'hard_negative'

    def sample_easy_negative_pair(self):
        phase_A, phase_B = np.random.choice(range(1, 9), size=2, replace=False)
        enso_A = np.random.choice(self.enso_categories)
        enso_B = np.random.choice(self.enso_categories)
        key_A, key_B = (phase_A, enso_A), (phase_B, enso_B)
        idx_A = (np.random.choice(self.index[key_A]) if self.index[key_A]
                 else self.labels[self.labels['bsiso_phase'] == phase_A].sample(1).index[0])
        idx_B = (np.random.choice(self.index[key_B]) if self.index[key_B]
                 else self.labels[self.labels['bsiso_phase'] == phase_B].sample(1).index[0])
        return idx_A, idx_B, 'easy_negative'

    def _random_category(self):
        valid = [k for k in self.index if len(self.index[k]) > 0]
        return valid[np.random.randint(len(valid))]


class BSISOPairDataset(Dataset):
    def __init__(self, X_data, labels_df, phase_enso_index,
                 mode='train', train_indices=None):
        self.X       = X_data
        self.labels  = labels_df
        self.sampler = PairSampler(labels_df, phase_enso_index)
        self.mode    = mode
        self.train_indices = (train_indices if train_indices is not None
                              else np.arange(len(X_data)))
        if mode == 'val':
            self.val_pairs = self._create_val_pairs()

    def _create_val_pairs(self):
        pairs = []
        val_labels = self.labels.loc[self.train_indices]
        for phase in range(1, 9):
            for enso in self.sampler.enso_categories:
                group = val_labels[
                    (val_labels['bsiso_phase'] == phase) &
                    (val_labels['enso_category'] == enso)
                ].index.tolist()
                for i in range(len(group)):
                    for j in range(i + 1, len(group)):
                        pairs.append((group[i], group[j], 'positive'))
        return pairs[:1000]

    def __len__(self):
        return (len(self.train_indices) if self.mode == 'train'
                else len(self.val_pairs))

    def __getitem__(self, idx):
        if self.mode == 'train':
            r = np.random.rand()
            if r < 0.30:
                idx_A, idx_B, _ = self.sampler.sample_positive_pair()
            elif r < 0.50:
                idx_A, idx_B, _ = self.sampler.sample_hard_negative_pair()
            else:
                idx_A, idx_B, _ = self.sampler.sample_easy_negative_pair()
        else:
            idx_A, idx_B, _ = self.val_pairs[idx]
        field_A = torch.from_numpy(self.X[idx_A]).float()
        field_B = torch.from_numpy(self.X[idx_B]).float()
        return field_A, field_B


train_dataset = BSISOPairDataset(X, labels, phase_enso_index, mode='train', train_indices=train_idx)
val_dataset   = BSISOPairDataset(X, labels, phase_enso_index, mode='val',   train_indices=val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train dataset: {len(train_dataset)} items  →  {len(train_loader)} batches/epoch')
print(f'Val dataset:   {len(val_dataset)} pairs   →  {len(val_loader)} batches')

## Cell 4 — CNN Encoder + InfoNCE Loss

Identical to notebook 07. `embedding_dim=2`, L2-normalized output → unit circle in R².

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, embedding_dim=2):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc          = nn.Linear(128, embedding_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).view(x.size(0), -1)
        z = self.fc(x)
        z = F.normalize(z, p=2, dim=1)
        return z


def InfoNCE_loss(z_A, z_B, temperature):
    sim_matrix = torch.matmul(z_A, z_B.T) / temperature
    labels_ = torch.arange(z_A.size(0), device=z_A.device)
    return F.cross_entropy(sim_matrix, labels_)

print('CNNEncoder + InfoNCE_loss defined.')

## Cell 5 — Training + Diagnostics Function (one τ per call)

Factored as a function so the sweep loop in Cell 6 stays tight. Returns a dict of metrics for the comparison table.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import cross_val_score, GroupKFold
from tqdm.notebook import tqdm

year_groups = labels['date'].dt.year.values

def train_one_tau(tau):
    """Train + evaluate the 2D encoder at the given temperature. Returns metrics dict."""
    tag = tau_tag(tau)
    out_dir = f'{PROJECT_DIR}/results/lee_2d_{tag}'
    os.makedirs(out_dir, exist_ok=True)

    print(f'\n{"="*70}\n  τ = {tau:.2f}  ({tag})\n{"="*70}')

    encoder   = CNNEncoder(embedding_dim=EMBEDDING_DIM).to(device)
    optimizer = optim.Adam(encoder.parameters(), lr=LR, weight_decay=0)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    history = {'train_loss': [], 'val_loss': [], 'epoch_time': []}

    for epoch in range(EPOCHS):
        t0 = time.time()
        # Train
        encoder.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f'τ={tau:.2f}  ep {epoch+1}/{EPOCHS}', leave=False)
        for fA, fB in pbar:
            fA = fA.to(device, non_blocking=True)
            fB = fB.to(device, non_blocking=True)
            zA = encoder(fA); zB = encoder(fB)
            loss = InfoNCE_loss(zA, zB, temperature=tau)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        train_loss /= len(train_loader)

        # Val
        encoder.eval()
        val_loss = 0.0
        with torch.no_grad():
            for fA, fB in val_loader:
                fA = fA.to(device, non_blocking=True)
                fB = fB.to(device, non_blocking=True)
                zA = encoder(fA); zB = encoder(fB)
                val_loss += InfoNCE_loss(zA, zB, temperature=tau).item()
        val_loss /= len(val_loader)
        scheduler.step()

        et = time.time() - t0
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['epoch_time'].append(et)
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f'  ep {epoch+1:2d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    # Save model + history
    torch.save(encoder.state_dict(), f'{CHECKPOINT_DIR}/encoder_2d_lee_{tag}_final.pth')
    with open(f'{CHECKPOINT_DIR}/training_history_2d_lee_{tag}.json', 'w') as f:
        json.dump(history, f, indent=2)

    # Training curves
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.plot(history['train_loss'], label='Train', linewidth=2)
    ax.plot(history['val_loss'],   label='Val',   linewidth=2)
    ax.axhline(y=np.log(64), color='gray', linestyle='--', alpha=0.5, label='log(64) baseline')
    ax.set_xlabel('Epoch'); ax.set_ylabel('InfoNCE Loss')
    ax.set_title(f'Training Curves  (τ={tau:.2f})', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Extract embeddings
    encoder.eval()
    embeddings_2d = np.zeros((len(X), EMBEDDING_DIM), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(X), 128):
            end = min(start + 128, len(X))
            batch = torch.from_numpy(X[start:end]).float().to(device)
            embeddings_2d[start:end] = encoder(batch).cpu().numpy()
    np.save(f'{out_dir}/embeddings.npy', embeddings_2d)

    # Angular spread (collapse diagnostic)
    angles = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
    angular_spread = float(angles.max() - angles.min())
    print(f'  Angular spread: {angular_spread:.3f} rad  (full circle = {2*np.pi:.3f})')

    # 2D scatter (val)
    Z_val = embeddings_2d[val_idx]
    lv = labels.loc[val_idx]
    phase_colors = plt.cm.tab10(np.linspace(0, 0.8, 8))
    enso_palette = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
    enso_marker  = {'El Nino': '^',       'Neutral': 'o',        'La Nina': 's'}
    theta = np.linspace(0, 2*np.pi, 200)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    ax = axes[0]
    for ph in range(1, 9):
        m = lv['bsiso_phase'] == ph
        ax.scatter(Z_val[m, 0], Z_val[m, 1], c=[phase_colors[ph-1]], s=18, alpha=0.7, label=f'Phase {ph}')
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=0.8)
    ax.set_title(f'τ={tau:.2f}  by BSISO Phase', fontweight='bold')
    ax.set_xlabel('z₁'); ax.set_ylabel('z₂'); ax.set_aspect('equal')
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.2)
    ax = axes[1]
    for cat in ['El Nino', 'Neutral', 'La Nina']:
        m = lv['enso_category'] == cat
        ax.scatter(Z_val[m, 0], Z_val[m, 1], c=enso_palette[cat], marker=enso_marker[cat], s=18, alpha=0.5, label=cat)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=0.8)
    ax.set_title(f'τ={tau:.2f}  by ENSO', fontweight='bold')
    ax.set_xlabel('z₁'); ax.set_ylabel('z₂'); ax.set_aspect('equal')
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    ax.legend(); ax.grid(alpha=0.2)
    plt.suptitle(f'2D Embedding (val), τ={tau:.2f},  spread={angular_spread:.2f} rad', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{out_dir}/embedding_2d_overview.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Linear probes
    Z_train = embeddings_2d[train_idx]
    Z_val_  = embeddings_2d[val_idx]
    gkf = GroupKFold(n_splits=5)

    # BSISO phase
    y_tr = labels.loc[train_idx, 'bsiso_phase'].values
    y_va = labels.loc[val_idx,   'bsiso_phase'].values
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(Z_train, y_tr)
    phase_val_acc = float(accuracy_score(y_va, clf.predict(Z_val_)))
    cv_phase = cross_val_score(clf, embeddings_2d, labels['bsiso_phase'].values,
                               cv=gkf, groups=year_groups, scoring='accuracy', n_jobs=-1)
    phase_cv_mean = float(cv_phase.mean()); phase_cv_std = float(cv_phase.std())

    # ENSO balanced
    y_tr = labels.loc[train_idx, 'enso_category'].values
    y_va = labels.loc[val_idx,   'enso_category'].values
    clf_bal = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced')
    clf_bal.fit(Z_train, y_tr)
    enso_bal_val = float(balanced_accuracy_score(y_va, clf_bal.predict(Z_val_)))
    cv_enso = cross_val_score(clf_bal, embeddings_2d, labels['enso_category'].values,
                              cv=gkf, groups=year_groups, scoring='balanced_accuracy', n_jobs=-1)
    enso_cv_mean = float(cv_enso.mean()); enso_cv_std = float(cv_enso.std())

    probe_results = {
        'BSISO Phase':              {'val_acc': phase_val_acc, 'cv_mean': phase_cv_mean, 'cv_std': phase_cv_std},
        'ENSO Category (balanced)': {'val_bal_acc': enso_bal_val, 'cv_mean': enso_cv_mean, 'cv_std': enso_cv_std},
    }
    with open(f'{out_dir}/linear_probe_results.json', 'w') as f:
        json.dump(probe_results, f, indent=2)
    print(f'  BSISO val acc:  {phase_val_acc*100:.1f}%   CV: {phase_cv_mean*100:.1f}% ± {phase_cv_std*100:.1f}%')
    print(f'  ENSO bal-acc:   {enso_bal_val*100:.1f}%   CV: {enso_cv_mean*100:.1f}% ± {enso_cv_std*100:.1f}%')

    # ENSO displacement z-score
    disp_mag = []
    for phase in range(1, 9):
        mEN = (labels['bsiso_phase'] == phase) & (labels['enso_category'] == 'El Nino')
        mLN = (labels['bsiso_phase'] == phase) & (labels['enso_category'] == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3:
            disp_mag.append(np.nan); continue
        cEN = embeddings_2d[mEN].mean(axis=0); cLN = embeddings_2d[mLN].mean(axis=0)
        disp_mag.append(np.linalg.norm(cEN - cLN))
    rng = np.random.default_rng(42)
    baseline_mag = []
    for _ in range(100):
        shuf = labels['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values
        mtrl = []
        for phase in range(1, 9):
            mph = (labels['bsiso_phase'] == phase).values
            mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
            if mEN.sum() < 3 or mLN.sum() < 3: continue
            mtrl.append(np.linalg.norm(embeddings_2d[mEN].mean(axis=0) - embeddings_2d[mLN].mean(axis=0)))
        if mtrl: baseline_mag.append(np.mean(mtrl))
    bmu = float(np.mean(baseline_mag)); bsd = float(np.std(baseline_mag))
    obs_mu = float(np.nanmean(disp_mag))
    z_score = float((obs_mu - bmu) / (bsd + 1e-8))

    fig, ax = plt.subplots(figsize=(8, 5))
    valid_phases = [p for p, m in zip(range(1, 9), disp_mag) if not np.isnan(m)]
    valid_mags   = [m for m in disp_mag if not np.isnan(m)]
    ax.bar(valid_phases, valid_mags, color='steelblue', alpha=0.8)
    ax.axhline(bmu, color='red', linestyle='--', linewidth=1.5, label=f'Null mean ({bmu:.4f})')
    ax.axhline(bmu + 2*bsd, color='red', linestyle=':', linewidth=1, label='Null +2σ')
    ax.axhline(obs_mu, color='steelblue', linewidth=2, label=f'Observed mean ({obs_mu:.4f})')
    ax.set_xticks(range(1, 9))
    ax.set_xlabel('BSISO Phase'); ax.set_ylabel('||EN−LN||')
    ax.set_title(f'ENSO Displacement, τ={tau:.2f},  z={z_score:.2f}  (64D baseline: 3.83)', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/enso_displacement.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f'  z-score: {z_score:.2f}')

    return {
        'tau': tau,
        'tag': tag,
        'final_train_loss':  float(history['train_loss'][-1]),
        'final_val_loss':    float(history['val_loss'][-1]),
        'phase_val_acc':     phase_val_acc,
        'phase_cv_mean':     phase_cv_mean,
        'phase_cv_std':      phase_cv_std,
        'enso_bal_val':      enso_bal_val,
        'enso_cv_mean':      enso_cv_mean,
        'enso_cv_std':       enso_cv_std,
        'z_score':           z_score,
        'angular_spread':    angular_spread,
        'embeddings_path':   f'{out_dir}/embeddings.npy',
    }

print('train_one_tau() defined.')

## Cell 6 — Run the Sweep

~30 min per τ × 3 τ values = ~90 min. Get a coffee.

In [ ]:
results = []
t_sweep_start = time.time()
for tau in TAU_VALUES:
    res = train_one_tau(tau)
    results.append(res)
print(f'\nTotal sweep time: {(time.time() - t_sweep_start)/60:.1f} min')

## Cell 7 — Comparison Table (4-way: τ=0.07 from notebook 07 + 3 new)

Loads the existing `results/lee_2d/` artifacts (the failed τ=0.07 run from notebook 07) so the comparison includes all four τ values.

In [ ]:
# --- Load τ=0.07 result from notebook 07's output dir, if present ---
legacy_dir = f'{PROJECT_DIR}/results/lee_2d'
legacy_present = os.path.exists(f'{legacy_dir}/linear_probe_results.json') and \
                 os.path.exists(f'{legacy_dir}/embeddings.npy')

if legacy_present:
    with open(f'{legacy_dir}/linear_probe_results.json') as f:
        legacy_probe = json.load(f)
    legacy_emb = np.load(f'{legacy_dir}/embeddings.npy')
    legacy_angles = np.arctan2(legacy_emb[:, 1], legacy_emb[:, 0])
    legacy_spread = float(legacy_angles.max() - legacy_angles.min())

    # Recompute z-score from legacy embeddings (same logic as in train_one_tau)
    disp_mag = []
    for phase in range(1, 9):
        mEN = (labels['bsiso_phase'] == phase) & (labels['enso_category'] == 'El Nino')
        mLN = (labels['bsiso_phase'] == phase) & (labels['enso_category'] == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: disp_mag.append(np.nan); continue
        cEN = legacy_emb[mEN].mean(axis=0); cLN = legacy_emb[mLN].mean(axis=0)
        disp_mag.append(np.linalg.norm(cEN - cLN))
    rng = np.random.default_rng(42)
    baseline_mag = []
    for _ in range(100):
        shuf = labels['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values
        mtrl = []
        for phase in range(1, 9):
            mph = (labels['bsiso_phase'] == phase).values
            mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
            if mEN.sum() < 3 or mLN.sum() < 3: continue
            mtrl.append(np.linalg.norm(legacy_emb[mEN].mean(axis=0) - legacy_emb[mLN].mean(axis=0)))
        if mtrl: baseline_mag.append(np.mean(mtrl))
    legacy_z = float((np.nanmean(disp_mag) - np.mean(baseline_mag)) / (np.std(baseline_mag) + 1e-8))

    legacy_row = {
        'tau': 0.07,
        'tag': 'tau007',
        'final_train_loss': None,
        'final_val_loss':   None,
        'phase_val_acc':    legacy_probe['BSISO Phase']['val_acc'],
        'phase_cv_mean':    legacy_probe['BSISO Phase']['cv_mean'],
        'phase_cv_std':     legacy_probe['BSISO Phase']['cv_std'],
        'enso_bal_val':     legacy_probe['ENSO Category (balanced)']['val_bal_acc'],
        'enso_cv_mean':     legacy_probe['ENSO Category (balanced)']['cv_mean'],
        'enso_cv_std':      legacy_probe['ENSO Category (balanced)']['cv_std'],
        'z_score':          legacy_z,
        'angular_spread':   legacy_spread,
        'embeddings_path':  f'{legacy_dir}/embeddings.npy',
    }
    all_results = [legacy_row] + results
    print(f'Loaded τ=0.07 legacy result: spread={legacy_spread:.3f} rad, z={legacy_z:.2f}')
else:
    all_results = results
    print('No legacy τ=0.07 result found — comparison includes only new sweep values.')

# Build comparison table
rows = []
for r in all_results:
    rows.append({
        'τ':                f"{r['tau']:.2f}",
        'phase val acc':    f"{r['phase_val_acc']*100:.1f}%",
        'phase 5-fold CV':  f"{r['phase_cv_mean']*100:.1f}% ± {r['phase_cv_std']*100:.1f}%",
        'ENSO bal-acc val': f"{r['enso_bal_val']*100:.1f}%",
        'ENSO bal CV':      f"{r['enso_cv_mean']*100:.1f}% ± {r['enso_cv_std']*100:.1f}%",
        'z-score':          f"{r['z_score']:.2f}",
        'angular spread':   f"{r['angular_spread']:.2f} rad",
        'final train loss': f"{r['final_train_loss']:.4f}" if r['final_train_loss'] is not None else 'n/a',
        'final val loss':   f"{r['final_val_loss']:.4f}"   if r['final_val_loss']   is not None else 'n/a',
    })

df_comp = pd.DataFrame(rows)
print('\n' + '=' * 110)
print('TEMPERATURE SWEEP COMPARISON  (64D Lee MJJAS baseline: phase 67.7%, z 3.83)')
print('=' * 110)
print(df_comp.to_string(index=False))

# Save CSV
df_comp.to_csv(f'{SWEEP_DIR}/comparison_table.csv', index=False)
print(f'\nSaved: results/lee_2d_tau_sweep/comparison_table.csv')

## Cell 8 — 4-Panel Scatter Figure (one panel per τ, val set, colored by phase)

Visually compare collapse-vs-spread across τ values. Each panel includes the unit circle and the angular spread in the title.

In [ ]:
n = len(all_results)
ncols = min(n, 4); nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows), squeeze=False)
phase_colors = plt.cm.tab10(np.linspace(0, 0.8, 8))
theta = np.linspace(0, 2*np.pi, 200)

for i, r in enumerate(all_results):
    ax = axes[i // ncols, i % ncols]
    emb = np.load(r['embeddings_path'])
    Z_val_p = emb[val_idx]
    lv = labels.loc[val_idx]
    for ph in range(1, 9):
        m = lv['bsiso_phase'] == ph
        ax.scatter(Z_val_p[m, 0], Z_val_p[m, 1], c=[phase_colors[ph-1]], s=10, alpha=0.6, label=f'P{ph}')
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=0.6)
    ax.set_title(f"τ={r['tau']:.2f}   spread={r['angular_spread']:.2f} rad   z={r['z_score']:.2f}\n"
                 f"phase val={r['phase_val_acc']*100:.1f}%",
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
    ax.set_aspect('equal'); ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    ax.grid(alpha=0.2)
    if i == 0:
        ax.legend(fontsize=7, ncol=2, loc='center')

# Hide any unused subplots
for j in range(n, nrows*ncols):
    axes[j // ncols, j % ncols].axis('off')

plt.suptitle('2D Encoder Temperature Sweep — Val Embeddings (colored by BSISO Phase)\n'
             '64D baseline: phase val 67.7%, z=3.83',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SWEEP_DIR}/scatter_4panel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lee_2d_tau_sweep/scatter_4panel.png')

## Cell 9 — Auto-Decision + Summary Markdown

Writes `temperature_sweep_summary.md` with the comparison table and the recommendation:
- **Greenlight Phase 2** at the best τ if any τ achieves val ≥ 62% AND z ≥ 3.0.
- **All-collapse fallback** (max angular spread < 1.0 rad) → temperature isn't the issue; recommend Option B (drop L2 norm) or Option C (Phase 4 dim sweep).
- **Partial recovery** — pick best by composite ranking and document caveats.

In [ ]:
# --- Decision logic ---
# Hard pass: phase_val_acc >= 0.62 AND z_score >= 3.0
hard_pass = [r for r in all_results if r['phase_val_acc'] >= 0.62 and r['z_score'] >= 3.0]
max_spread = max(r['angular_spread'] for r in all_results)
all_collapsed = max_spread < 1.0   # < ~57° of arc

if hard_pass:
    best = max(hard_pass, key=lambda r: (r['phase_val_acc'], r['z_score']))
    decision = 'GREENLIGHT_PHASE_2'
    decision_text = (
        f"**Best τ = {best['tau']:.2f}** (phase val {best['phase_val_acc']*100:.1f}%, "
        f"z={best['z_score']:.2f}, spread {best['angular_spread']:.2f} rad).\n\n"
        f"This τ meets both decision thresholds (val ≥ 62% AND z ≥ 3.0). "
        f"→ **Use τ = {best['tau']:.2f} for Phase 2** (SSL temporal model)."
    )
elif all_collapsed:
    decision = 'TEMPERATURE_NOT_THE_ISSUE'
    decision_text = (
        f"All τ values collapsed (max angular spread = {max_spread:.2f} rad < 1.0). "
        f"Temperature is not the bottleneck. → **Recommend Option B (drop L2 normalization) "
        f"or Option C (Phase 4 dimension sweep).**"
    )
else:
    # Partial: pick best by composite (phase_acc + z/10 + spread/(2π)) — soft ranking
    best = max(all_results, key=lambda r: r['phase_val_acc'] + r['z_score']/10 + r['angular_spread']/(2*np.pi))
    decision = 'PARTIAL_RECOVERY'
    decision_text = (
        f"**Partial recovery.** No τ met both thresholds, but the best result is τ = {best['tau']:.2f} "
        f"(phase val {best['phase_val_acc']*100:.1f}%, z={best['z_score']:.2f}, "
        f"spread {best['angular_spread']:.2f} rad). "
        f"→ Discuss with advisor before greenlighting Phase 2; options remain to drop L2 norm or sweep dim."
    )

print('=' * 70)
print(f'DECISION: {decision}')
print('=' * 70)
print(decision_text)

# --- Write summary markdown ---
summary_md = f"""# Phase 1 — Temperature Sweep Summary

**Auto-generated by notebook 07b.**  
**Date:** {time.strftime('%Y-%m-%d')}  
**Goal:** test whether a softer InfoNCE temperature recovers the 2D encoder from the τ=0.07 collapse seen in notebook 07.

## Comparison Table

{df_comp.to_markdown(index=False)}

**64D Lee MJJAS baseline** (year-based split): phase val acc 67.7%, ENSO displacement z-score 3.83.

## Decision

{decision_text}

## Diagnostic notes

- **Angular spread** is the range (max−min) of embedding angles in radians. Full circle = 2π ≈ 6.28. Values < 1.0 rad indicate the model collapsed to a small arc.
- **Hard-pass thresholds:** phase val ≥ 62% AND z ≥ 3.0 (set in `extension_2d_plan.md`).
- **All-collapse threshold:** max angular spread < 1.0 rad → temperature isn't the issue.

## Files produced

Per τ (tag = `tau{{int(τ × 100):03d}}`):
- `checkpoints/encoder_2d_lee_{{tag}}_final.pth`
- `checkpoints/training_history_2d_lee_{{tag}}.json`
- `results/lee_2d_{{tag}}/embeddings.npy`
- `results/lee_2d_{{tag}}/training_curves.png`
- `results/lee_2d_{{tag}}/embedding_2d_overview.png`
- `results/lee_2d_{{tag}}/linear_probe_results.json`
- `results/lee_2d_{{tag}}/enso_displacement.png`

Aggregate:
- `results/lee_2d_tau_sweep/comparison_table.csv`
- `results/lee_2d_tau_sweep/scatter_4panel.png`
- `results/lee_2d_tau_sweep/temperature_sweep_summary.md` (this file)
"""

with open(f'{SWEEP_DIR}/temperature_sweep_summary.md', 'w') as f:
    f.write(summary_md)
print(f'\nSaved: {SWEEP_DIR}/temperature_sweep_summary.md')

print(f'\nAll outputs in {SWEEP_DIR}/:')
for f in sorted(os.listdir(SWEEP_DIR)):
    mb = os.path.getsize(f'{SWEEP_DIR}/{f}') / 1e6
    print(f'  {f}  ({mb:.2f} MB)')

## Cell 10 — (Optional) Download All Sweep Outputs to Local Machine

In [ ]:
from google.colab import files
for fname in sorted(os.listdir(SWEEP_DIR)):
    files.download(f'{SWEEP_DIR}/{fname}')

# Optionally also download per-τ embeddings + scatter for the winning τ later

---
## Done!

**Send back to me:**
1. `temperature_sweep_summary.md`
2. `comparison_table.csv`
3. `scatter_4panel.png`

I'll log the result into `extension_2d_plan.md`, update the Phase 1 status, and either start Phase 2 (SSL temporal at the winning τ) or pivot to Option B / Phase 4 dim sweep depending on what the table says.

---
*DDCS Project | jh9141@nyu.edu*